## EDA Notebook
This notebook will contain the eda process of all the raw csv files

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
from pathlib import Path

In [2]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",50)
pd.set_option("display.float_format","{:.2f}".format)

In [3]:
# Setting paths for easier navigation
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [4]:
# Plot Style
sns.set_theme(style="darkgrid",palette="muted")
COLORS = {"Fatal": "#e74c3c", "Serious": "#e67e22", "Minor": "#3498db"}

### Loading the datasets

In [8]:
# India mendeley - data/raw/crashes/csv
mendeley_path = RAW / "india_mendeley" / "crashes.csv"
df_mendeley = pd.read_csv(mendeley_path)
print(f"df_mendeley shape - {df_mendeley.shape}")
df_mendeley.head(3)

df_mendeley shape - (2898, 17)


,S. No.,Month,Crash Date,Crash Day,Article Date,Location,Million Plus City,State,LatLong,Vehicle 1,Vehicle/Object 2,Killed,Injured,Age,Gender,Road Type,Crash Type
0,1,January,44561,Friday,44562,Alni,Nil,Maharashtra,"18.278457450374876, 76.00957388786227",Car,Tractor,4,0,"70,50,45,23",Both,NH,Head On Collision
1,2,January,44561,Friday,44562,Hariharganj,Nil,Jharkhand,"24.543681356839087, 84.28057070288547",Pick Up,Truck,6,18,NaN,Both,NH,Head On Collision
2,3,January,44560,Thursday,44562,Dabolim,Nil,Goa,"15.390068035586195, 73.85497360324335",Car,Divider,3,2,"27,24,24",Male,NH,Fixed Object Collision


In [9]:
# Kaggle india accidents
kaggle_path = RAW / "kaggle_india" / "accident_prediction_india.csv"
df_kaggle = pd.read_csv(kaggle_path)
print(f"india road accidents shape - {df_kaggle.shape}")
df_kaggle.head(3)

india road accidents shape - (3000, 22)


,State Name,City Name,Year,Month,Day of Week,Time of Day,Accident Severity,Number of Vehicles Involved,Vehicle Type Involved,Number of Casualties,Number of Fatalities,Weather Conditions,Road Type,Road Condition,Lighting Conditions,Traffic Control Presence,Speed Limit (km/h),Driver Age,Driver Gender,Driver License Status,Alcohol Involvement,Accident Location Details
0,Jammu and Kashmir,Unknown,2021,May,Monday,1:46,Serious,5,Cycle,0,4,Hazy,National Highway,Wet,Dark,Signs,61,66,Male,NaN,Yes,Curve
1,Uttar Pradesh,Lucknow,2018,January,Wednesday,21:30,Minor,5,Truck,5,4,Hazy,Urban Road,Dry,Dusk,Signs,92,60,Male,NaN,Yes,Straight Road
2,Chhattisgarh,Unknown,2023,May,Wednesday,5:37,Minor,5,Pedestrian,6,5,Foggy,National Highway,Under Construction,Dawn,Signs,120,26,Female,NaN,No,Bridge


In [11]:
global_path = RAW / "global" / "Global Road Accidents Dataset.csv"
df_global = pd.read_csv(global_path, low_memory=False)
print(f"Global accident shape - {df_global.shape}")
df_global.head(3)

Global accident shape - (132000, 30)


,Country,Year,Month,Day of Week,Time of Day,Urban/Rural,Road Type,Weather Conditions,Visibility Level,Number of Vehicles Involved,Speed Limit,Driver Age Group,Driver Gender,Driver Alcohol Level,Driver Fatigue,Vehicle Condition,Pedestrians Involved,Cyclists Involved,Accident Severity,Number of Injuries,Number of Fatalities,Emergency Response Time,Traffic Volume,Road Condition,Accident Cause,Insurance Claims,Medical Cost,Economic Loss,Region,Population Density
0,USA,2002,October,Tuesday,Evening,Rural,Street,Windy,220.41,1,37,18-25,Male,0.05,0,Poor,1,2,Moderate,8,2,58.63,7412.75,Wet,Weather,4,40499.86,22072.88,Europe,3866.27
1,UK,2014,December,Saturday,Evening,Urban,Street,Windy,168.31,3,96,18-25,Female,0.23,1,Poor,1,1,Minor,6,1,58.04,4458.63,Snow-covered,Mechanical Failure,3,6486.60,9534.40,North America,2333.92
2,USA,2012,July,Sunday,Afternoon,Urban,Highway,Snowy,341.29,4,62,41-60,Male,0.14,0,Moderate,0,0,Moderate,13,4,42.37,9856.92,Wet,Speeding,4,29164.41,58009.15,South America,4408.89


In [12]:
# MoRTH files
morth_dir = RAW / "morth"

df_sw_acc  = pd.read_csv(morth_dir / "statewise_accidents_2019_2023.csv")
df_sw_fat  = pd.read_csv(morth_dir / "statewise_fatalities_2019_2023.csv")
df_collide = pd.read_csv(morth_dir / "collision_types_2023.csv")
df_violate = pd.read_csv(morth_dir / "violation_types_2023.csv")
df_cities  = pd.read_csv(morth_dir / "large_cities_2023.csv")

print(f"  statewise_accidents   → {df_sw_acc.shape}")
print(f"  statewise_fatalities  → {df_sw_fat.shape}")
print(f"  collision_types       → {df_collide.shape}")
print(f"  violation_types       → {df_violate.shape}")
print(f"  large_cities          → {df_cities.shape}")

  statewise_accidents   → (40, 14)
  statewise_fatalities  → (40, 13)
  collision_types       → (19, 10)
  violation_types       → (13, 10)
  large_cities          → (51, 14)


### Data Quality Audit

In [14]:
def audit(df, name):
    print(f"\n{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    print(f"  Shape      : {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  Duplicates : {df.duplicated().sum():,}")
    print(f"  Dtypes     :\n{df.dtypes.value_counts().to_string()}")
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if len(miss):
        print(f"  Missing values:")
        for col, cnt in miss.items():
            print(f"    {col:<35} {cnt:>5} ({cnt/len(df)*100:.1f}%)")
    else:
        print("  Missing values : None")

audit(df_mendeley, "India Mendeley")


 India Mendeley
  Shape      : 2,898 rows × 17 cols
  Duplicates : 0
  Dtypes     :
str      13
int64     4
  Missing values:
    Age                                   728 (25.1%)
    Gender                                263 (9.1%)


In [15]:
audit(df_kaggle,   "Kaggle India")


 Kaggle India
  Shape      : 3,000 rows × 22 cols
  Duplicates : 0
  Dtypes     :
str      16
int64     6
  Missing values:
    Traffic Control Presence              716 (23.9%)
    Driver License Status                 975 (32.5%)


In [16]:
audit(df_global,   "Global Kaggle")


 Global Kaggle
  Shape      : 132,000 rows × 30 cols
  Duplicates : 0
  Dtypes     :
str        14
int64       9
float64     7
  Missing values : None


In [20]:
# Column overview for each dataset
def column_overview(data_name, data_given):
    print(f"\n— {data_name} columns")
    print("-" * 30)

    for col in data_given.columns:
        dtype = str(data_given[col].dtype)
        nuniq = data_given[col].nunique()
        sample = str(data_given[col].dropna().iloc[0])[:40] if len(data_given[col].dropna()) > 0 else "N/A"

        print(f"{col:<40} {dtype:<10} {nuniq:>6} unique | e.g. {sample}")

column_overview("Mendeley", df_mendeley)


— Mendeley columns
------------------------------
S. No.                                   int64        2898 unique | e.g. 1
Month                                    str            12 unique | e.g. January
Crash Date                               str           698 unique | e.g. 44561
Crash Day                                str             7 unique | e.g. Friday
Article Date                             int64         728 unique | e.g. 44562
Location                                 str          2725 unique | e.g. Alni
Million Plus City                        str            43 unique | e.g. Nil
State                                    str            30 unique | e.g. Maharashtra
LatLong                                  str          2898 unique | e.g. 18.278457450374876, 76.00957388786227
Vehicle 1                                str            17 unique | e.g. Car
Vehicle/Object 2                         str            39 unique | e.g. Tractor
Killed                                   int64

In [21]:
column_overview("Kaggle India", df_kaggle)


— Kaggle India columns
------------------------------
State Name                               str            32 unique | e.g. Jammu and Kashmir
City Name                                str            28 unique | e.g. Unknown
Year                                     int64           6 unique | e.g. 2021
Month                                    str            12 unique | e.g. May
Day of Week                              str             7 unique | e.g. Monday
Time of Day                              str          1263 unique | e.g. 1:46
Accident Severity                        str             3 unique | e.g. Serious
Number of Vehicles Involved              int64           5 unique | e.g. 5
Vehicle Type Involved                    str             7 unique | e.g. Cycle
Number of Casualties                     int64          11 unique | e.g. 0
Number of Fatalities                     int64           6 unique | e.g. 4
Weather Conditions                       str             5 unique | e.g. Ha

In [22]:
column_overview("Global", df_global)


— Global columns
------------------------------
Country                                  str            10 unique | e.g. USA
Year                                     int64          25 unique | e.g. 2002
Month                                    str            12 unique | e.g. October
Day of Week                              str             7 unique | e.g. Tuesday
Time of Day                              str             4 unique | e.g. Evening
Urban/Rural                              str             2 unique | e.g. Rural
Road Type                                str             3 unique | e.g. Street
Weather Conditions                       str             5 unique | e.g. Windy
Visibility Level                         float64    132000 unique | e.g. 220.4146505416504
Number of Vehicles Involved              int64           4 unique | e.g. 1
Speed Limit                              int64          90 unique | e.g. 37
Driver Age Group                         str             5 unique | e.g. 